# Session 4 — Bandits: an RL-based agent

**Salih Tutun, PhD** · DAT 5668 · Module 2 · Practice session (about 80 minutes)

I am Salih Tutun. Until I say **open the notebook**, this file stays closed. **Do not scroll.** Pairs talk first. Then type. After we run: **LLM**, **RL**, or **hybrid** — and one line for your project.

---

## Where we are (say this with me)

| Session | Family | What you can already do |
| --- | --- | --- |
| 1 | Both | Name $s$, $a$, $r$, $s'$. A chatbot that only writes a paragraph is **not** an agent. |
| 2 | LLM | Tool loop on a refund desk: look up, then refund / refuse / escalate. |
| 3 | LLM | Retrieve a policy chunk (RAG), hand off, reviewer releases on a **real CFPB** ticket. |
| **4 (tonight)** | **RL** | **Bandit: one scarce Amazon FBA restock slot, three ASINs.** |
| 5 (next) | RL | MDP: inventory that **remembers** stock. |

Tonight is the **first RL practice**. Same agent loop as Sessions 1–3. Different family.

**Family this session: RL-based agent.** Repeated numeric choices and a reward. Session 3's document agent does not fit this job. This bandit does not fit Session 3's documents. Do not mix them because both are "AI." Pick the family from the work.

![A bandit fills one Amazon FBA restock slot](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-bandit.png)

![The agent chooses a; the environment returns r and s'](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-agent-env.png)

---

## Real business problem — Amazon FBA restock priority

An **Amazon FBA** (Fulfillment by Amazon) desk can fill **one** scarce **restock-priority slot** each business day. Capacity is one. Three products want that slot.

**What is an ASIN?** **Amazon Standard Identification Number** — Amazon's product ID for a listing (like a SKU on their marketplace). In this notebook we use three **instructional** labels built from public invoice product lines — not confidential Amazon catalog IDs:

| ASIN (course label) | Public product line behind it |
| --- | --- |
| `ASIN-HOME` | Home and lifestyle |
| `ASIN-ELEC` | Electronic accessories |
| `ASIN-SPORT` | Sports and travel |

Only the ASIN you chose pays that day's **contribution** (sales total). You never see the other two. That is a **multi-armed bandit**.

**Why this is still a bandit — not Session 5.** We do **not** track units on hand. Choosing HOME today does not change tomorrow's stock state in this model. Session 5 adds that memory (MDP).

**Real data (Google Drive in Colab).** I share `MyDrive/DAT5668/data/` with the class. Daily contributions come from the public supermarket invoice file (`supermarket-sales.csv`), also shipped as `amazon-fba-restock-daily.csv`. No employer data. No API key required for this lab. (Optional free Gemini is for LLM decide steps in other sessions — not needed tonight.)

| Word | Tonight (say it out loud) |
| --- | --- |
| **AGENT** | ε-greedy (the **bandit**) chooses *a* |
| **ENVIRONMENT** | That ASIN's daily sales contribution |
| State *s* | This visit: which ASIN gets the restock slot |
| Observe | Only the contribution of the ASIN you picked |
| Decide | ε-greedy. RAG does **not** belong here. |
| Action *a* | ASIN-HOME, ASIN-ELEC, or ASIN-SPORT |
| Reward *r* | That ASIN's sales that day (can be 0 on a quiet day) |
| Next *s′* | Tomorrow's visit |

---

## The rule we will code (board version)

**ε-greedy (decide).** Draw a number uniformly between 0 and 1.
- If it is **below** `epsilon` → return a **random** ASIN index (**explore**).
- Otherwise → return the smallest index of the largest running mean (**exploit**). Use `np.argmax`.

| epsilon | Behavior | Business read |
| --- | --- | --- |
| 0 | Never explores | Locks onto an early lucky ASIN |
| 1 | Never exploits | Never uses what it learned |
| 0.10 | Mostly exploit, still checks | Useful middle for tonight |

**Sample mean (learn from r).** After reward `r` on ASIN index `a`:
- `counts[a] = counts[a] + 1`
- `values[a] = values[a] + (r - values[a]) / counts[a]`

**Regret (evaluation).** Extra contribution the best **mean** ASIN would have earned, minus what you got. Code can be correct and regret still high — that is a **finding**, not a bug.

---

## How slides and this notebook work together

| Cue | What you do |
| --- | --- |
| **OPEN** | I said open. Work only the stop on the slide. |
| **RUN WITH ME** | Already written. Run together. Do not edit. |
| **JUMP BACK** | Eyes on the projector. Do not scroll ahead. |

## Run of show
1. **Slides first** — Recap Sessions 1–3. ε-greedy on the board. Notebook closed.
2. **OPEN Stops 1–3** — Name the desk. Load real ASIN days. Read day 0.
3. **STAY Stop 4** — Write ε-greedy until PASS.
4. **STAY Stops 5–6** — Practice with me: try several epsilons and seeds, watch regret and counts change, then vote.
5. **JUMP BACK** — Debrief on slides. Then OPEN for the framework line.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll ahead of the stop I named.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **OPEN** | I opened this stop from the slides. |
| **PAIR (N min)** | Groups of 2 (or 3). One laptop. Talk the full time. Do not type until I say. |
| **YOU FILL THIS** | Guess to a neighbor. Two people share. Then type `YOU TYPE HERE`. |
| **TODO** | You write the function **with your pair**. Search for `TODO`. We stay until every line says PASS. |
| **RUN WITH ME** | Already written. Run it. Do not type in that cell. |
| **TRY WITH ME** | Change a number I name. Re-run. Say out loud what changed. |
| **JUMP BACK** | Eyes on slides. Do not scroll. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** — and one line for your project framework. |


## Stop 1 — OPEN · PAIR (3 min). Name the loop before any code

> **OPEN** from slides: Stops 1–3. **STOP. Do not scroll.**

**What I want you to understand.** Before any Python, name the loop for **tonight's Amazon FBA desk** — not Session 3's CFPB ticket. If you cannot name $s$, $a$, $r$, $s'$, you are not ready to code.

**Hints (after you talk, not before):**
- State: today's visit — which ASIN gets the one restock-priority slot
- Actions: ASIN-HOME, ASIN-ELEC, ASIN-SPORT
- Reward: that ASIN's sales that day (others stay hidden)
- Next: tomorrow's visit (no stock memory tonight)

Groups of 2 (or 3). One laptop. Talk the full 3 minutes. Then type.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Pair first (3 min). Then type. Short phrases for the Amazon FBA desk.
state_tonight = ""       # YOU TYPE HERE — what the desk sees before it picks
action_set = ""          # YOU TYPE HERE — the three ASINs
reward_tonight = ""      # YOU TYPE HERE — what comes back after the close
next_state = ""          # YOU TYPE HERE — what happens tomorrow
print("state:", state_tonight or "(talk, then type)")
print("actions:", action_set or "(talk, then type)")
print("reward:", reward_tonight or "(talk, then type)")
print("next:", next_state or "(talk, then type)")


## Before Stop 2 — Google Drive data (Colab)

**How this class uses data.** I put the practice CSVs in a shared **Google Drive** folder. You open the notebook in Colab, mount Drive once, and load from that folder — same path for everyone.

**Folder we use:**
```text
MyDrive/DAT5668/data/
  amazon-fba-restock-daily.csv
  supermarket-sales.csv
```

**Steps (do this with me):**
1. Accept the Drive share link for `DAT5668` (Viewer is enough).
2. In Colab, run the **mount** cell below. Approve access when Google asks.
3. Confirm `DRIVE_DATA_FOLDER` in the next load cell points to `/content/drive/MyDrive/DAT5668/data`.
4. Run the load cell. The printout must show that path (or a local fallback if you are offline on the repo).

If Drive is not mounted yet, the load cell will tell you which paths it tried.


In [ ]:
# RUN WITH ME in Google Colab — mount Drive once per runtime.
# Local Jupyter: you can skip this cell; the load cell will use assets/data instead.
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/DAT5668/data")

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted at /content/drive")
except ImportError:
    print("Not Colab — skip mount. Load cell will use local assets/data if present.")

if DRIVE_ROOT.exists():
    files = sorted(p.name for p in DRIVE_ROOT.glob("*.csv"))
    print("Found DAT5668/data with", len(files), "CSV(s):", files)
    need = ["amazon-fba-restock-daily.csv", "supermarket-sales.csv"]
    missing = [n for n in need if not (DRIVE_ROOT / n).exists()]
    if missing:
        print("MISSING on Drive (ask instructor to share):", missing)
    else:
        print("Session 4 files are on Drive. Continue to the load cell.")
else:
    print("Folder not found yet:", DRIVE_ROOT)
    print("Mount Drive and make sure DAT5668/data is under MyDrive (shared with you).")


## Stop 2 — RUN WITH ME. Real ASIN daily contributions (from Drive or local)

> **RUN WITH ME.** I run. You watch. Then look at the printed table together.

**Teaching points while it runs:**
1. Prefer **Google Drive** `DAT5668/data` in Colab so everyone uses the same files I shared.
2. These are **real public invoice days**, remapped to three ASINs. Not a random toy table.
3. The printout of **oracle mean** is for **evaluation only** (regret later). The agent does **not** get those means inside decide.
4. Day 0 is the first calendar day in the table. We will ask who "won" that day next.
5. A quiet day can show `0.00` for an ASIN — that is still a real observation if you chose it.

No Gemini key tonight. This lab is RL on numbers.


In [ ]:
# RUN WITH ME. Load real ASIN days — Google Drive (Colab) or local course files.
# No Gemini key needed for this RL lab.
#
# === How we use Google Drive in this course ===
# 1) Instructor shares a Drive folder with the class (Viewer is enough), e.g.
#      MyDrive/DAT5668/data/
#    Files inside that folder:
#      amazon-fba-restock-daily.csv
#      supermarket-sales.csv   (fallback builder)
# 2) In Colab: run the mount cell (or mount once per runtime).
# 3) Set DRIVE_DATA_FOLDER below to that folder path, then run this cell.
# 4) If Drive is empty / not mounted, we fall back to the repo assets/data paths.
from pathlib import Path
import csv
from collections import defaultdict
from datetime import datetime
import numpy as np

OFFERS = ["ASIN-HOME", "ASIN-ELEC", "ASIN-SPORT"]
LINE_TO_ASIN = {
    "Home and lifestyle": "ASIN-HOME",
    "Electronic accessories": "ASIN-ELEC",
    "Sports and travel": "ASIN-SPORT",
}

# <<< SET THIS when using Google Drive in Colab (leave "" to use local/repo files) >>>
# Example:
# DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"
DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"


def course_data(name):
    """Find a practice CSV: Google Drive first (if set), then local course paths."""
    candidates = []
    if DRIVE_DATA_FOLDER.strip():
        candidates.append(Path(DRIVE_DATA_FOLDER.strip()) / name)
    # Common Colab default for this course (works even if DRIVE_DATA_FOLDER was cleared)
    candidates.append(Path("/content/drive/MyDrive/DAT5668/data") / name)
    candidates.extend(
        [
            Path("data") / name,
            Path("../../assets/data") / name,
            Path("assets/data") / name,
            Path("/content/data") / name,
        ]
    )
    tried = []
    for path in candidates:
        tried.append(str(path))
        if path.exists():
            return path
    raise FileNotFoundError(
        name
        + " not found. Mount Google Drive and put the CSV in DAT5668/data — or open the notebook from the course repo. Tried:\n- "
        + "\n- ".join(tried)
    )


def load_from_restock_csv():
    path = course_data("amazon-fba-restock-daily.csv")
    rows = list(csv.DictReader(path.open(encoding="utf-8")))
    dates = [row["date"] for row in rows]
    rewards = {asin: [float(row[asin]) for row in rows] for asin in OFFERS}
    return dates, rewards, f"{path} (invoice-derived ASIN days)"


def load_from_supermarket_invoices():
    path = course_data("supermarket-sales.csv")
    daily = defaultdict(lambda: {asin: 0.0 for asin in OFFERS})
    with path.open(encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            line = row["Product line"].strip()
            if line not in LINE_TO_ASIN:
                continue
            day = datetime.strptime(row["Date"].strip(), "%m/%d/%Y").date().isoformat()
            daily[day][LINE_TO_ASIN[line]] += float(row["Total"])
    dates = sorted(daily)
    rewards = {asin: [daily[day][asin] for day in dates] for asin in OFFERS}
    return dates, rewards, f"{path} → daily ASIN contribution"


def load_asin_rewards():
    try:
        return load_from_restock_csv()
    except FileNotFoundError:
        return load_from_supermarket_invoices()


DATES, REWARDS, DATA_SOURCE = load_asin_rewards()
N_DAYS = len(DATES)
# Oracle for evaluation only — NEVER feed this into epsilon_greedy.
TRUE_MEAN = {asin: float(np.mean(REWARDS[asin])) for asin in OFFERS}


def asin_reward(offer, day):
    """Environment step: return only the chosen ASIN's contribution that day."""
    return REWARDS[offer][day % N_DAYS]


def best_possible_rate():
    """Best average daily contribution — used to score regret, not to pick actions."""
    return max(TRUE_MEAN.values())


print("Business problem: Amazon FBA — one restock-priority slot per day")
print("Data source:", DATA_SOURCE)
print("DRIVE_DATA_FOLDER:", DRIVE_DATA_FOLDER or "(not set — using first path that exists)")
print("Calendar:", DATES[0], "→", DATES[-1], "| days:", N_DAYS)
print("ASINs:", OFFERS)
print()
print("First 5 daily contributions (talk about these next):")
print(f"{'day':>3} {'date':>12} {'HOME':>10} {'ELEC':>10} {'SPORT':>10}")
for day in range(min(5, N_DAYS)):
    print(
        f"{day:3d} {DATES[day]:>12} "
        f"{REWARDS['ASIN-HOME'][day]:10.2f} {REWARDS['ASIN-ELEC'][day]:10.2f} {REWARDS['ASIN-SPORT'][day]:10.2f}"
    )
print()
print("Oracle mean daily contribution (evaluation only):", {k: round(v, 2) for k, v in TRUE_MEAN.items()})
print("Oracle best mean ASIN:", max(TRUE_MEAN, key=TRUE_MEAN.get))
print("We will solve: which epsilon should this FBA desk use on these real days?")


## Stop 3 — OPEN · PAIR (3 min). Day 0 only

> **OPEN** (still Stops 1–3). Look at **day 0** in the printout above.

**Why this stop matters.** A greedy desk that "just picks yesterday's winner forever" is ε = 0. That is a common business failure. I want you to see it on real numbers before you code.

Talk with your pair:

1. Which ASIN had the best contribution on day 0?
2. If the desk is greedy forever after day 0, what goes wrong?

Then type. **Do not open the TODO yet.**

*(After you guess: day 0 is usually ASIN-ELEC on this file. Oracle mean over the whole quarter is often ASIN-SPORT — different question.)*


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Pair first (3 min). Allowed ASINs: ASIN-HOME, ASIN-ELEC, ASIN-SPORT.
day0_best_guess = ""  # YOU TYPE HERE — which ASIN won day 0?
greedy_risk = ""      # YOU TYPE HERE — one sentence: what goes wrong if we never explore?
print("Day 0 winner (your read):", day0_best_guess or "(look at the table, then type)")
print("Greedy risk:", greedy_risk or "(one sentence)")


## Stop 4 — STAY · PAIR (20 min). Write the bandit until PASS

> **STAY in the notebook** (slides: Stop 4 until PASS). Search for `TODO`. One laptop. Talk, then type together.

**What you are building (say it before you type):**
1. `epsilon_greedy` — the **decide** box. Explore or exploit.
2. Sample-mean update inside `run_bandit` — learn from the reward you actually saw.
3. Run the **check** cell. Stay until every line says **PASS**.

**Common mistakes I will walk for:**
- Forgetting that `np.argmax` on a tie returns the **smallest** index
- Updating `values` without first bumping `counts` (or dividing by the old count)
- Leaving `return 0` in `epsilon_greedy` so every day picks HOME

You are training on **real** ASIN daily contributions — the Amazon FBA desk from the slides.


In [ ]:
# TODO — you write this with your pair. Search for TODO.
# Teaching note (read before you type):
#   epsilon_greedy = the DECIDE box.
#   Explore with probability epsilon: pick a random ASIN index.
#   Exploit otherwise: pick np.argmax(values) (smallest index on a tie).
#   After the environment returns reward r, update ONLY that ASIN:
#       counts[a] += 1
#       values[a] += (r - values[a]) / counts[a]
#   That is the sample-mean update. You do not store every past day.
def epsilon_greedy(values, epsilon, rng):
    """Return an action index. Explore with probability epsilon; else exploit."""
    # TODO: if rng.random() < epsilon, return a random index in 0 .. len(values)-1
    # TODO: otherwise return the smallest index of the largest value (np.argmax)
    return 0


def run_bandit(epsilon, n_visits=None, seed=7):
    """Fill one restock-priority slot each day. Only the chosen ASIN is observed."""
    n_visits = n_visits or N_DAYS
    rng = np.random.default_rng(seed)  # same seed → fair epsilon comparisons
    values = np.zeros(len(OFFERS))     # running mean contribution per ASIN
    counts = np.zeros(len(OFFERS))     # how many times we chose each ASIN
    rewards = []
    for day in range(n_visits):
        index = epsilon_greedy(values, epsilon, rng)  # decide
        offer = OFFERS[index]
        reward = asin_reward(offer, day)              # environment returns r
        # TODO: update counts[index] and values[index] with the sample-mean formula
        # counts[index] = ...
        # values[index] = ...
        rewards.append(reward)
    return values, counts, np.cumsum(rewards)


values, counts, cumulative = run_bandit(0.10)
print("values", np.round(values, 2))
print("counts", counts.astype(int))
print("cumulative contribution", round(float(cumulative[-1]), 2) if len(cumulative) else 0)
print("If counts are all zero, the update TODO is still empty — keep going.")


**This cell — the checks.** **RUN WITH ME** after your pair finishes the TODOs.

Do not edit the expected behavior. If you see `NOT YET`, fix the TODO above and re-run **both** the TODO cell and this check cell.

When you see three `PASS` lines, raise a hand. Then we compare epsilons.


In [ ]:
# RUN WITH ME — do not edit. Re-run until every line says PASS.
# What these checks teach:
#   1) epsilon=0 must exploit the unique max, and break ties toward the smallest index.
#   2) epsilon=1 must explore (all three indices appear).
#   3) With all-zero values, argmax picks index 0 → epsilon=0 locks on ASIN-HOME forever.
#      That is a BUSINESS failure pattern, even when the code is correct.
#   4) epsilon=0.10 on seed 7 still samples every ASIN at least once.
def check_bandit():
    rng = np.random.default_rng(0)
    if epsilon_greedy(np.array([0.1, 0.5, 0.2]), 0.0, rng) != 1:
        print("NOT YET epsilon_greedy — with epsilon=0 pick the largest value")
        return
    if epsilon_greedy(np.array([0.5, 0.5, 0.1]), 0.0, rng) != 0:
        print("NOT YET epsilon_greedy — ties break toward the smallest index (np.argmax)")
        return
    picks = {epsilon_greedy(np.array([0.9, 0.1, 0.0]), 1.0, rng) for _ in range(80)}
    if picks != {0, 1, 2}:
        print("NOT YET epsilon_greedy — with epsilon=1 return a random index each time")
        return
    values, counts, cumulative = run_bandit(0.0, seed=7)
    if counts.sum() == 0:
        print("NOT YET run_bandit — write the sample-mean update (counts and values)")
        return
    if int(counts[0]) != N_DAYS or int(counts[1]) != 0 or int(counts[2]) != 0:
        print("NOT YET run_bandit — epsilon=0 with seed 7 should lock on ASIN-HOME (index 0) all days")
        return
    expected = float(np.mean(REWARDS["ASIN-HOME"]))
    if abs(float(values[0]) - expected) > 1e-9:
        print("NOT YET run_bandit — values[0] should be the sample mean of ASIN-HOME rewards")
        return
    values10, counts10, _ = run_bandit(0.10, seed=7)
    if counts10.min() < 1:
        print("NOT YET — epsilon=0.10 should still sample every ASIN at least once with seed 7")
        return
    print("PASS epsilon_greedy")
    print("PASS sample-mean update")
    print("PASS epsilon=0 locks on ASIN-HOME; epsilon=0.10 still explores")
    print("counts at 0.10:", counts10.astype(int).tolist(), "| learned:", OFFERS[int(np.argmax(values10))])
    print("Desk check: you trained on", N_DAYS, "real ASIN days from", DATA_SOURCE.split()[0])


check_bandit()


## Stop 5 — STAY · PRACTICE WITH ME. Try settings. Watch what changes.

> **STAY.** This is the learning block. We do **not** run once and leave.

**How we practice (say it with me):**
1. I name an experiment.
2. You change **one** number.
3. Re-run that cell.
4. Say out loud: what moved — `learned`, `counts`, or `regret`?
5. Change the number again. Compare.

**What the numbers mean:**
- `learned` — ASIN with the highest running mean at the end (what a greedy desk would pick now)
- `counts` — how often each ASIN got the scarce restock slot (exploration footprint)
- `regret` — contribution left on the table vs always picking the best **mean** ASIN (oracle)

If counts are all zero, go back and finish the TODOs in Stop 4.


In [ ]:
# RUN WITH ME after PASS. Helpers for every practice trial below.
# Regret = (best mean ASIN's contribution) − (what your policy earned).
def regret_so_far(cumulative):
    visits = np.arange(1, len(cumulative) + 1)
    best = best_possible_rate() * visits
    return best - cumulative


def practice_run(epsilon, seed=7, n_visits=None, label=""):
    """One trial you can re-run after changing epsilon or seed. Watch counts and regret."""
    values, counts, cumulative = run_bandit(epsilon, n_visits=n_visits, seed=seed)
    if counts.sum() == 0:
        print(f"{label}epsilon={epsilon} seed={seed}: NOT YET — finish the TODOs above first")
        return None
    gap = float(regret_so_far(cumulative)[-1])
    learned = OFFERS[int(np.argmax(values))]
    tag = f"{label}" if label else ""
    print(
        f"{tag}epsilon={epsilon}  seed={seed}  days={int(counts.sum())}  "
        f"learned={learned}  counts={counts.astype(int).tolist()}  "
        f"cumulative={float(cumulative[-1]):.1f}  regret={gap:.1f}"
    )
    return {"learned": learned, "counts": counts.astype(int), "regret": gap, "cumulative": float(cumulative[-1])}


def report(epsilon, seed=7):
    """Same as practice_run — kept so later cells that call report(...) still work."""
    return practice_run(epsilon, seed=seed)


print("Oracle best mean ASIN:", max(TRUE_MEAN, key=TRUE_MEAN.get), {k: round(v, 2) for k, v in TRUE_MEAN.items()})
print("Helpers ready. We will try several settings next — change a number, re-run, watch what moves.")


### Experiment A — RUN WITH ME. Three epsilons, same seed

I run first. You watch. Then we say together which desk is too shy (ε=0) and which is restless (ε=0.40).


In [ ]:
# RUN WITH ME — Experiment A: three epsilons, same seed.
# Look at counts (exploration footprint) and regret. Say which looks too shy / too restless.
print("=== Experiment A: epsilon in {0, 0.10, 0.40}, seed=7 ===")
for epsilon in (0.0, 0.10, 0.40):
    practice_run(epsilon, seed=7)
print("Talk: what moved from 0 → 0.10 → 0.40?")


### Experiment B — TRY WITH ME. You pick epsilon

Change `try_epsilon`. Re-run. Do this **at least twice** (for example 0.05 and 0.20). Do not scroll until you have two printouts.


In [ ]:
# TRY WITH ME — Experiment B: YOU pick an epsilon.
# 1) Change try_epsilon (try 0.05, then 0.20, then 0.50).
# 2) Re-run this cell each time.
# 3) Write one line below on what changed (learned ASIN? counts? regret?).
try_epsilon = 0.10  # <<< CHANGE ME, then re-run
practice_run(try_epsilon, seed=7, label="your try: ")
print("Re-run with another epsilon before you scroll. Learning = watching the numbers move.")


### Experiment C — TRY WITH ME. Same epsilon, different seed

Luck early in the quarter can change what locks in. Keep epsilon = 0.10. Change only `try_seed`. Re-run at least twice.


In [ ]:
# TRY WITH ME — Experiment C: same epsilon, different luck (seed).
# Early random pulls change what a greedy streak locks onto.
# Change try_seed (7 → 1 → 99). Keep epsilon fixed. Re-run.
fixed_epsilon = 0.10   # keep this fixed while you change the seed
try_seed = 7           # <<< CHANGE ME, then re-run
practice_run(fixed_epsilon, seed=try_seed, label="seed trial: ")
print("Ask: did learned ASIN or regret jump when only the seed changed?")


### Experiment D — TRY WITH ME. Short desk vs full quarter

A pilot that only runs 10 days is not the same as 89 days. Change `short_days`. Re-run.


In [ ]:
# TRY WITH ME — Experiment D: short horizon vs full quarter.
# Same epsilon, same seed. Fewer days = less time to explore.
# Change short_days (try 10, then 30, then N_DAYS). Re-run.
short_epsilon = 0.10
short_days = 10  # <<< CHANGE ME (10, 30, or N_DAYS)
practice_run(short_epsilon, seed=7, n_visits=short_days, label="short run: ")
print("Full quarter has", N_DAYS, "days. What happens to regret when the desk only runs", short_days, "days?")


### Write what you saw — PAIR then type

Talk 60 seconds. Then type. Use the printouts from Experiments A–D. Guess before you polish the sentence.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# After Experiments A–D. At least 40 characters each.
what_changed_epsilon = ""  # YOU TYPE HERE — what moved when YOU changed epsilon?
what_changed_seed = ""     # YOU TYPE HERE — what moved when only the seed changed?
what_changed_horizon = ""  # YOU TYPE HERE — short run vs full quarter?
for label, text in (
    ("epsilon trials", what_changed_epsilon),
    ("seed trials", what_changed_seed),
    ("horizon trials", what_changed_horizon),
):
    if len(text) < 40:
        print("NOT YET —", label, "(need 40+ characters)")
    else:
        print(label + ":", text)


## Stop 6 — STAY · PAIR then class vote. Pick an epsilon for this FBA desk

> **STAY.** You just practiced. Now decide as a manager.

Neighbor 20 seconds. Two voices. Using what you saw in Experiments A–D:
- 0 never explores → locks early
- 1 never settles → never uses learning
- **0.10** is the useful middle for tonight

Agree on a number for **this Amazon FBA desk**, type it, run.

> After you run: **JUMP BACK** to the slides for debrief. Do **not** scroll to Stop 7 until I say OPEN again.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. A number from 0 to 1. This is the desk decision after practice.
class_epsilon = None  # YOU TYPE HERE, for example 0.10
if class_epsilon is None:
    print("Agree on an epsilon, type it, and run again.")
else:
    practice_run(class_epsilon, seed=7, label="class vote: ")
    print("Say whether this was too shy or too restless — use your experiment printouts.")
    print("JUMP BACK to slides for debrief — do not scroll to Stop 7 yet.")


## Stop 7 — OPEN after slides · PAIR (4 min). Debrief

> **JUMP BACK** first (slides). After walking-out answers on the projector, **OPEN** this stop and type.

**Walking-out answers (we say them together, then you type short lines):**

1. ε = 0 never retries the ASINs it skipped. An early unlucky week locks the desk onto the wrong ASIN — you saw a version of this when seed or epsilon changed.
2. A CFPB dispute needs a citation, a refuse option, and a human release — **not** a numeric pull. Wrong family.
3. Do not bandit a decision where one bad action is unsafe: medical denial, large wire, auto-refund that can empty the till.

**Big line for the term:** Code can be correct and regret still high. Trying multiple settings is how you *see* that. That is a finding, not a bug.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Pair first. Short answers in your own words.
epsilon0_fails_because = ""  # YOU TYPE HERE
cfpb_is_wrong_job_because = ""  # YOU TYPE HERE
would_not_bandit = ""  # YOU TYPE HERE — e.g. large wire / medical denial / auto-refund
print("1.", epsilon0_fails_because or "(talk, then type)")
print("2.", cfpb_is_wrong_job_because or "(talk, then type)")
print("3.", would_not_bandit or "(talk, then type)")


## Add this to your framework — OPEN · STOP. Do not scroll.

> **OPEN** from the framework slide.

Guess to a neighbor (20 seconds): after today, is **your** project **LLM**, **RL**, or **hybrid**? Two voices share. Then type one sentence you will keep all term.

**Tonight's stamp:**
- Amazon FBA restock slot on real ASIN days → **RL** (bandit)
- Session 3 CFPB complaint → **LLM**
- Hybrid later: learned ASIN on the usual day; a language agent on the exception email

**Before you pack up:** Homework 2 adds a fourth ASIN (`ASIN-BEAUTY`) — same ε-greedy rule; keep trying settings until you can explain what changed. Session 5: supermarket shelf as an MDP — a bandit has no stock memory.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")
